# 03 - Silver Layer Transformation

## Purpose

This notebook transforms validated Bronze census tables into clean, standardized Silver tables.

### Responsibilities

- Read validated Bronze tables
- Remove non-business metadata rows
- Enrich records with business metadata
- Add audit information
- Preserve occupational hierarchy
- Write Delta tables into the Silver schema

### Input

bronze.st_main_workers

bronze.sc_main_workers

bronze.industrial_main_workers

bronze.industrial_marginal_workers

### Output

silver.st_main_workers

silver.sc_main_workers

silver.industrial_main_workers

silver.industrial_marginal_workers

### 1 — Imports

In [0]:
from pyspark.sql.functions import (
    col,
    when,
    current_timestamp,
    lit
)

## Common Transformation Functions

These reusable functions standardize the enrichment logic across all census tables.

Occupational tables (SC/ST) and Industrial tables have different schemas,
therefore separate transformation functions are used.

### 2 — Occupational Transformation Function

In [0]:
def transform_occupational_table(df, community_category):
    """
    Silver transformation for:

    - st_main_workers
    - sc_main_workers
    """

    return (

        df

        # -----------------------------
        # Community Metadata
        # -----------------------------
        .withColumn(
            "Community_Category",
            lit(community_category)
        )

        # -----------------------------
        # Geographic Metadata
        # -----------------------------
        .withColumn(

            "Geo_Level",

            when(
                col("District_Code") == 0,
                "STATE_TOTAL"
            )

            .otherwise(
                "DISTRICT_DETAIL"
            )

        )

        # -----------------------------
        # Occupational Hierarchy
        # -----------------------------
        .withColumn(

            "Hierarchy_Level",

            when(

                (col("Division") == "0") &
                (col("Sub_Division") == "00") &
                (col("Group") == "000") &
                (col("Family") == "0000"),

                "GRAND_TOTAL"

            )

            .when(

                col("Sub_Division") == "00",

                "DIVISION_TOTAL"

            )

            .when(

                col("Group") == "000",

                "SUBDIVISION_TOTAL"

            )

            .when(

                col("Family") == "0000",

                "GROUP_TOTAL"

            )

            .otherwise(

                "FAMILY_DETAIL"

            )

        )

        # -----------------------------
        # Audit Columns
        # -----------------------------
        .withColumn(
            "Data_Layer",
            lit("Silver")
        )

        .withColumn(
            "Silver_Load_Time",
            current_timestamp()
        )

    )

### New columns 
To make downstream analysis easier

| Column             | Purpose                           |
| ------------------ | --------------------------------- |
| Community_Category | Scheduled Tribe / Scheduled Caste |
| Geo_Level          | State Total or District Detail    |
| Hierarchy_Level    | Explicit occupational hierarchy   |
| Data_Layer         | Data lineage                      |
| Silver_Load_Time   | ETL audit timestamp               |


## Industrial Table Transformation

Industrial tables contain the same occupational hierarchy but
measure workers by industrial sector rather than rural/urban counts.

These tables also contain five metadata rows in the Bronze layer,
which are removed here because they are documentation rather than
business records.

### 3 — Industrial Transformation Function

In [0]:
def transform_industrial_table(df, worker_type):
    """
    Silver transformation for:

    - industrial_main_workers
    - industrial_marginal_workers
    """

    # -------------------------------------------------------
    # Remove metadata/documentation rows
    # -------------------------------------------------------

    df = df.filter(
    col("Division").isNotNull()
)

    # -------------------------------------------------------
    # Enrichment
    # -------------------------------------------------------

    df = (

        df

        .withColumn(

            "Worker_Type",

            lit(worker_type)

        )

        .withColumn(

            "Geo_Level",

            when(

                col("District_Code") == 0,

                "STATE_TOTAL"

            )

            .otherwise(

                "DISTRICT_DETAIL"

            )

        )

        .withColumn(

            "Hierarchy_Level",

            when(

                (col("Division") == "0") &
                (col("Sub_Division") == "00"),

                "GRAND_TOTAL"

            )

            .when(

                col("Sub_Division") == "00",

                "DIVISION_TOTAL"

            )

            .otherwise(

                "SUBDIVISION_DETAIL"

            )

        )

        .withColumn(

            "Data_Layer",

            lit("Silver")

        )

        .withColumn(

            "Silver_Load_Time",

            current_timestamp()

        )

    )

    return df

### 4 — Helper Function to Save Silver Tables

In [0]:
def write_to_silver(df, table_name):

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(
            f"pyspark_dbt.silver.{table_name}"
        )
    )

    print(f"✓ Saved: silver.{table_name}")

## Part 1 Complete

The notebook now contains reusable transformation functions.

The next section applies these functions to:

- ST Main Workers
- SC Main Workers

and writes the transformed data into the Silver schema.

## Occupational Tables

The occupational tables share an identical schema and hierarchy structure.

The Silver transformation enriches them with:

- Community Category
- Geographic Level
- Hierarchy Level
- Audit Metadata

### 5 — Load ST Main Workers Bronze Table

In [0]:
print("=" * 70)
print("PROCESSING ST MAIN WORKERS")
print("=" * 70)

df_st_bronze = spark.read.table(
    "pyspark_dbt.bronze.st_main_workers"
)

print(f"Bronze Rows: {df_st_bronze.count()}")

### 6 — Transform ST Main Workers

In [0]:
df_st_silver = transform_occupational_table(
    df_st_bronze,
    "Scheduled Tribe (ST)"
)

### 7 — Quick Verification

In [0]:
display(

    df_st_silver.select(

        "State_District",
        "Division",
        "Sub_Division",
        "Group",
        "Family",
        "Hierarchy_Level",
        "Geo_Level",
        "Community_Category"

    )

)

### 8 — Write ST Silver Table

In [0]:
write_to_silver(
    df_st_silver,
    "st_main_workers"
)

### Scheduled Caste Main Workers

### 9 — Load SC Main Workers Bronze Table

In [0]:
print("=" * 70)
print("PROCESSING SC MAIN WORKERS")
print("=" * 70)

df_sc_bronze = spark.read.table(
    "pyspark_dbt.bronze.sc_main_workers"
)

print(f"Bronze Rows: {df_sc_bronze.count()}")

### 10 — Transform SC Main Workers

In [0]:
df_sc_silver = transform_occupational_table(
    df_sc_bronze,
    "Scheduled Caste (SC)"
)

### 11 — Quick Verification

In [0]:
display(

    df_sc_silver.select(

        "State_District",
        "Division",
        "Sub_Division",
        "Group",
        "Family",
        "Hierarchy_Level",
        "Geo_Level",
        "Community_Category"

    )

)

### 12 — Write SC Silver Table

In [0]:
write_to_silver(
    df_sc_silver,
    "sc_main_workers"
)

### 13 — Occupational Layer Summary


In [0]:
print("=" * 70)
print("OCCUPATIONAL TABLES COMPLETED")
print("=" * 70)

print(
    """
Completed Tables:

✓ st_main_workers
✓ sc_main_workers

Metadata Added:

✓ Community_Category
✓ Geo_Level
✓ Hierarchy_Level
✓ Data_Layer
✓ Silver_Load_Time
"""
)

In [0]:
spark.read.table(
    "pyspark_dbt.silver.st_main_workers"
).printSchema()

## Industrial Tables

Industrial tables classify occupations by industrial category
(A through U) and by Total/Rural/Urban population segments.

The Silver transformation performs:

- Metadata row removal
- Geographic enrichment
- Occupational hierarchy enrichment
- Worker type enrichment
- Audit column creation

### 14 — Load Industrial Main Workers

In [0]:
print("=" * 70)
print("PROCESSING INDUSTRIAL MAIN WORKERS")
print("=" * 70)

df_ind_main_bronze = spark.read.table(
    "pyspark_dbt.bronze.industrial_main_workers"
)

print(f"Bronze Rows: {df_ind_main_bronze.count()}")

### 15 — Transform Industrial Main Workers

In [0]:
df_ind_main_silver = transform_industrial_table(
    df_ind_main_bronze,
    "Main Worker"
)

### 16 — Verify Metadata Removal

In [0]:
print(
    f"Rows after cleaning: "
    f"{df_ind_main_silver.count()}"
)

### 17 — Quick Verification

In [0]:
display(

    df_ind_main_silver.select(

        "Area_Name",
        "Division",
        "Sub_Division",
        "Total_Rural_Urban",
        "Hierarchy_Level",
        "Geo_Level",
        "Worker_Type"

    )

)

### 18 — Write Industrial Main Workers Silver Table

In [0]:
write_to_silver(
    df_ind_main_silver,
    "industrial_main_workers"
)

### Industrial Marginal Workers

### 9 — Load Industrial Marginal Workers

In [0]:
print("=" * 70)
print("PROCESSING INDUSTRIAL MARGINAL WORKERS")
print("=" * 70)

df_ind_marg_bronze = spark.read.table(
    "pyspark_dbt.bronze.industrial_marginal_workers"
)

print(f"Bronze Rows: {df_ind_marg_bronze.count()}")

### 20 — Transform Industrial Marginal Workers

In [0]:
df_ind_marg_silver = transform_industrial_table(
    df_ind_marg_bronze,
    "Marginal Worker"
)

### 21 — Verification

In [0]:
display(

    df_ind_marg_silver.select(

        "Area_Name",
        "Division",
        "Sub_Division",
        "Total_Rural_Urban",
        "Hierarchy_Level",
        "Geo_Level",
        "Worker_Type"

    )

)

### 22 — Write Industrial Marginal Workers Silver Table

In [0]:
write_to_silver(
    df_ind_marg_silver,
    "industrial_marginal_workers"
)

### 23 — Industrial Summary

In [0]:
print("=" * 70)
print("INDUSTRIAL TABLES COMPLETED")
print("=" * 70)

print(
"""
Completed Tables:

✓ industrial_main_workers
✓ industrial_marginal_workers

Metadata Added:

✓ Worker_Type
✓ Geo_Level
✓ Hierarchy_Level
✓ Data_Layer
✓ Silver_Load_Time

Cleaning Performed:

✓ Removed 5 metadata rows from industrial_main_workers
"""
)